# 08 — Business Impact & Decision Policy

This notebook translates the fraud model into an operational decision framework.

The objectives are to:

- quantify fraud concentration under different review capacities,
- measure lift over the baseline fraud rate,
- compare review workload with fraud capture,
- define an interpretable review-priority policy,
- evaluate illustrative cost-sensitive scenarios without assuming real bank costs,
- summarize the final model as a risk-based decision-support system.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(
    r"D:\bank-account-fraud-decision-system"
)

data_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "fraud_modeling_dataset.parquet"
)

df = pd.read_parquet(data_path)

target = "fraud_bool"

train_df = df[
    df["month"].between(0, 4)
].copy()

calibration_df = df[
    df["month"] == 5
].copy()

strategy_df = df[
    df["month"] == 6
].copy()

test_df = df[
    df["month"] == 7
].copy()

In [2]:
from catboost import CatBoostClassifier

cat_features = [
    "payment_type",
    "employment_status",
    "housing_status",
    "source",
    "device_os"
]

X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_cal = calibration_df.drop(columns=[target])
y_cal = calibration_df[target]

X_strategy = strategy_df.drop(columns=[target])
y_strategy = strategy_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]

def prepare_catboost_data(X):
    X_cb = X.copy()

    for col in cat_features:
        X_cb[col] = (
            X_cb[col]
            .fillna("Missing")
            .astype(str)
        )

    return X_cb

X_train_cb = prepare_catboost_data(X_train)
X_cal_cb = prepare_catboost_data(X_cal)
X_strategy_cb = prepare_catboost_data(X_strategy)
X_test_cb = prepare_catboost_data(X_test)

final_cat_model = CatBoostClassifier(
    depth=5,
    learning_rate=0.05,
    iterations=631,
    loss_function="Logloss",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=False
)

final_cat_model.fit(
    X_train_cb,
    y_train,
    cat_features=cat_features
)

CatBoostClassifier(auto_class_weights='Balanced', depth=5, iterations=631, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=False)

In [3]:
#Platt scaling calibration

from sklearn.linear_model import LogisticRegression

eps = 1e-6

cal_raw_proba = final_cat_model.predict_proba(
    X_cal_cb
)[:, 1]

cal_raw_clipped = np.clip(
    cal_raw_proba,
    eps,
    1 - eps
)

cal_logits = np.log(
    cal_raw_clipped
    / (1 - cal_raw_clipped)
).reshape(-1, 1)

platt_calibrator = LogisticRegression()

platt_calibrator.fit(
    cal_logits,
    y_cal
)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [ ]:
#Helper function to get calibrated probabilities for any dataset

def calibrated_probability(model, calibrator, X_cb):
    raw_proba = model.predict_proba(
        X_cb
    )[:, 1]

    raw_clipped = np.clip(
        raw_proba,
        eps,
        1 - eps
    )

    logits = np.log(
        raw_clipped
        / (1 - raw_clipped)
    ).reshape(-1, 1)

    calibrated = calibrator.predict_proba(
        logits
    )[:, 1]

    return calibrated

In [6]:
#Get calibrated scores for month 6 (strategy) and month 7 (test) datasets

strategy_proba = calibrated_probability(
    final_cat_model,
    platt_calibrator,
    X_strategy_cb
)

test_proba = calibrated_probability(
    final_cat_model,
    platt_calibrator,
    X_test_cb
)

## 1. Review Capacity, Fraud Capture, and Lift

A fraud-review system should be evaluated by how effectively it concentrates fraud within a limited investigation capacity.

For each review budget, applications are ranked by calibrated fraud probability and the highest-risk applications are selected.

The analysis reports:

- review workload,
- number of fraud cases captured,
- precision,
- recall,
- lift relative to the underlying fraud prevalence.

In [ ]:
#Helper function to create a review capacity table

def review_capacity_table(
    y_true,
    probabilities,
    review_rates
):

    eval_df = pd.DataFrame({
        "actual": np.asarray(y_true),
        "probability": probabilities
    })

    eval_df = (
        eval_df
        .sort_values(
            "probability",
            ascending=False
        )
        .reset_index(drop=True)
    )

    baseline_fraud_rate = (
        eval_df["actual"].mean()
    )

    total_fraud = (
        eval_df["actual"].sum()
    )

    rows = []

    for rate in review_rates:

        n_review = int(
            np.ceil(
                len(eval_df) * rate
            )
        )

        reviewed = eval_df.head(
            n_review
        )

        fraud_found = int(
            reviewed["actual"].sum()
        )

        precision = (
            fraud_found / n_review
        )

        recall = (
            fraud_found / total_fraud
        )

        lift = (
            precision
            / baseline_fraud_rate
        )

        threshold = reviewed[
            "probability"
        ].iloc[-1]

        rows.append({
            "review_rate_pct":
                rate * 100,

            "applications_reviewed":
                n_review,

            "fraud_found":
                fraud_found,

            "precision_pct":
                precision * 100,

            "recall_pct":
                recall * 100,

            "lift":
                lift,

            "probability_threshold":
                threshold
        })

    return pd.DataFrame(rows)

In [9]:
#Month 6 strategy review capacity table

review_rates = [
    0.01,
    0.02,
    0.05,
    0.10,
    0.20
]

strategy_review_table = (
    review_capacity_table(
        y_strategy,
        strategy_proba,
        review_rates
    )
)

display(strategy_review_table)

,review_rate_pct,applications_reviewed,fraud_found,precision_pct,recall_pct,lift,probability_threshold
0,1.0,1082,296,27.356747,20.413793,20.407756,0.191828
1,2.0,2164,451,20.841035,31.103448,15.547125,0.121018
2,5.0,5409,729,13.477537,50.275862,10.054057,0.058206
3,10.0,10817,960,8.874919,66.206897,6.620567,0.029248
4,20.0,21634,1182,5.463622,81.517241,4.075787,0.012674


In [11]:
#month 7 test review capacity table

test_review_table = (
    review_capacity_table(
        y_test,
        test_proba,
        review_rates
    )
)

display(test_review_table)

,review_rate_pct,applications_reviewed,fraud_found,precision_pct,recall_pct,lift,probability_threshold
0,1.0,969,318,32.817337,22.268908,22.255808,0.153196
1,2.0,1937,492,25.400103,34.453782,17.225646,0.092531
2,5.0,4843,768,15.857939,53.781513,10.754415,0.041320
3,10.0,9685,971,10.025813,67.997199,6.799228,0.019950
4,20.0,19369,1169,6.035417,81.862745,4.093053,0.008505


### Review-Capacity Performance

The risk-ranking system remained stable in the out-of-time test period.

At a 5% review capacity, the month-6 strategy period captured 50.28% of all fraud with 13.48% precision and approximately 10.05× lift over the underlying fraud rate.

In the unseen month-7 period, the highest-risk 5% contained 53.78% of all fraud, with 15.86% precision and approximately 10.75× lift.

Performance remained strong across all evaluated review capacities, suggesting that the model consistently concentrates fraud cases near the top of its risk ranking.

The month-7 capacity analysis is used only as an out-of-time diagnostic. Probability thresholds derived from the final test period are not used to modify the previously frozen operating policy.

## 2. Workload vs Fraud-Capture Trade-off

Increasing review capacity captures more fraud but also increases investigation workload and reduces precision.

Because the dataset does not provide the institution's investigation cost, fraud-loss amount, or review-team capacity, no single review rate can be claimed to be economically optimal.

Instead, several operating scenarios are compared to support capacity-dependent decision making.

In [12]:
decision_policy = (
    strategy_review_table[
        [
            "review_rate_pct",
            "applications_reviewed",
            "fraud_found",
            "precision_pct",
            "recall_pct",
            "lift"
        ]
    ]
    .copy()
)

decision_policy["fraud_missed"] = (
    y_strategy.sum()
    - decision_policy["fraud_found"]
)

decision_policy["nonfraud_reviewed"] = (
    decision_policy["applications_reviewed"]
    - decision_policy["fraud_found"]
)

display(decision_policy)

,review_rate_pct,applications_reviewed,fraud_found,precision_pct,recall_pct,lift,fraud_missed,nonfraud_reviewed
0,1.0,1082,296,27.356747,20.413793,20.407756,1154,786
1,2.0,2164,451,20.841035,31.103448,15.547125,999,1713
2,5.0,5409,729,13.477537,50.275862,10.054057,721,4680
3,10.0,10817,960,8.874919,66.206897,6.620567,490,9857
4,20.0,21634,1182,5.463622,81.517241,4.075787,268,20452


## 3. Capacity-Dependent Decision Policies

No single review threshold is universally optimal because the appropriate operating point depends on investigator capacity, fraud-loss severity, and review cost.

Three illustrative policies are therefore defined:

- Conservative review policy: top 1% of applications
- Balanced review policy: top 5% of applications
- Aggressive review policy: top 10% of applications

These labels describe review capacity rather than the severity of individual fraud cases.

In [13]:
policy_table = pd.DataFrame({
    "Policy": [
        "Conservative",
        "Balanced",
        "Aggressive"
    ],

    "Review_Capacity_Pct": [
        1,
        5,
        10
    ],

    "Applications_Reviewed": [
        1082,
        5409,
        10817
    ],

    "Fraud_Captured": [
        296,
        729,
        960
    ],

    "Fraud_Recall_Pct": [
        20.413793,
        50.275862,
        66.206897
    ],

    "Precision_Pct": [
        27.356747,
        13.477537,
        8.874919
    ],

    "Lift": [
        20.407756,
        10.054057,
        6.620567
    ],

    "Nonfraud_Reviewed": [
        786,
        4680,
        9857
    ]
})

display(policy_table)

,Policy,Review_Capacity_Pct,Applications_Reviewed,Fraud_Captured,Fraud_Recall_Pct,Precision_Pct,Lift,Nonfraud_Reviewed
0,Conservative,1,1082,296,20.413793,27.356747,20.407756,786
1,Balanced,5,5409,729,50.275862,13.477537,10.054057,4680
2,Aggressive,10,10817,960,66.206897,8.874919,6.620567,9857


### Policy Interpretation

The Conservative policy minimizes investigation workload and provides the highest precision and lift, but captures only about one fifth of fraud cases.

The Balanced policy reviews approximately 5% of applications and captures about half of all fraud cases. It therefore provides a practical compromise between investigation workload and fraud coverage.

The Aggressive policy increases fraud recall to approximately 66%, but requires roughly twice the investigation workload of the Balanced policy and produces substantially more non-fraud reviews.

Because actual investigation costs, fraud losses, and staffing capacity are not available in the dataset, these scenarios should be interpreted as decision-support options rather than economically optimized policies.

## 4. Investigation Efficiency

Investigation efficiency measures how much fraud is captured relative to the proportion of applications manually reviewed.

A highly effective ranking model should capture a disproportionately large share of fraud within a small review population.

In [14]:
policy_table["Fraud_Capture_per_1pct_Review"] = (
    policy_table["Fraud_Recall_Pct"]
    / policy_table["Review_Capacity_Pct"]
)

display(
    policy_table[
        [
            "Policy",
            "Review_Capacity_Pct",
            "Fraud_Recall_Pct",
            "Fraud_Capture_per_1pct_Review",
            "Precision_Pct",
            "Lift"
        ]
    ]
)

,Policy,Review_Capacity_Pct,Fraud_Recall_Pct,Fraud_Capture_per_1pct_Review,Precision_Pct,Lift
0,Conservative,1,20.413793,20.413793,27.356747,20.407756
1,Balanced,5,50.275862,10.055172,13.477537,10.054057
2,Aggressive,10,66.206897,6.620690,8.874919,6.620567


### Investigation Efficiency Interpretation

Fraud-capture efficiency decreases as review capacity expands.

The Conservative policy captures approximately 20.4% of all fraud for each 1% of applications reviewed. The Balanced policy captures about 10.1% of fraud per 1% reviewed, while the Aggressive policy captures about 6.6%.

This demonstrates diminishing returns: broader investigation capacity increases total fraud recall, but each additional unit of review workload contributes progressively less incremental fraud capture.

## 5. Illustrative Cost-Sensitive Analysis

The dataset does not contain actual fraud-loss amounts, investigation costs, or institution-specific operating costs.

To demonstrate how the decision system could support economic decision making, a set of hypothetical cost scenarios is evaluated.

These values are illustrative only and should not be interpreted as real banking costs.

In [15]:
illustrative_scenarios = pd.DataFrame({
    "Scenario": [
        "Low Fraud Cost",
        "Moderate Fraud Cost",
        "High Fraud Cost"
    ],
    "Cost_per_Review": [
        5,
        5,
        5
    ],
    "Cost_per_Missed_Fraud": [
        100,
        500,
        1000
    ]
})

display(illustrative_scenarios)

,Scenario,Cost_per_Review,Cost_per_Missed_Fraud
0,Low Fraud Cost,5,100
1,Moderate Fraud Cost,5,500
2,High Fraud Cost,5,1000


In [16]:
cost_results = []

for _, policy in policy_table.iterrows():

    for _, scenario in illustrative_scenarios.iterrows():

        review_cost = (
            policy["Applications_Reviewed"]
            * scenario["Cost_per_Review"]
        )

        missed_fraud = (
            y_strategy.sum()
            - policy["Fraud_Captured"]
        )

        missed_fraud_cost = (
            missed_fraud
            * scenario["Cost_per_Missed_Fraud"]
        )

        total_cost = (
            review_cost
            + missed_fraud_cost
        )

        cost_results.append({
            "Policy": policy["Policy"],
            "Scenario": scenario["Scenario"],
            "Review_Cost": review_cost,
            "Missed_Fraud_Cost": missed_fraud_cost,
            "Total_Illustrative_Cost": total_cost
        })

cost_results_df = pd.DataFrame(
    cost_results
)

display(cost_results_df)

,Policy,Scenario,Review_Cost,Missed_Fraud_Cost,Total_Illustrative_Cost
0,Conservative,Low Fraud Cost,5410,115400,120810
1,Conservative,Moderate Fraud Cost,5410,577000,582410
2,Conservative,High Fraud Cost,5410,1154000,1159410
3,Balanced,Low Fraud Cost,27045,72100,99145
4,Balanced,Moderate Fraud Cost,27045,360500,387545
5,Balanced,High Fraud Cost,27045,721000,748045
6,Aggressive,Low Fraud Cost,54085,49000,103085
7,Aggressive,Moderate Fraud Cost,54085,245000,299085
8,Aggressive,High Fraud Cost,54085,490000,544085


In [17]:
cost_pivot = (
    cost_results_df
    .pivot(
        index="Policy",
        columns="Scenario",
        values="Total_Illustrative_Cost"
    )
)

display(cost_pivot)

Scenario,High Fraud Cost,Low Fraud Cost,Moderate Fraud Cost
Policy,,,
Aggressive,544085,103085,299085
Balanced,748045,99145,387545
Conservative,1159410,120810,582410


### Cost-Sensitive Policy Interpretation

The illustrative cost analysis shows that the preferred operating policy depends on the economic consequences assigned to missed fraud.

When the assumed cost of missed fraud is relatively low, the Balanced policy produces the lowest total illustrative cost because it limits investigation workload while still capturing approximately half of fraud cases.

As the assumed cost of missed fraud increases, the Aggressive policy becomes more favorable. Its larger review workload is offset by the reduction in missed fraud.

This analysis demonstrates that fraud-review thresholds should be selected using institution-specific investigation costs, fraud-loss severity, and operational capacity rather than predictive metrics alone.

The numerical values in this section are hypothetical and are used only to demonstrate the decision framework.

In [18]:
best_policy_by_scenario = (
    cost_results_df
    .loc[
        cost_results_df.groupby("Scenario")[
            "Total_Illustrative_Cost"
        ].idxmin(),
        [
            "Scenario",
            "Policy",
            "Total_Illustrative_Cost"
        ]
    ]
    .reset_index(drop=True)
)

display(best_policy_by_scenario)

,Scenario,Policy,Total_Illustrative_Cost
0,High Fraud Cost,Aggressive,544085
1,Low Fraud Cost,Balanced,99145
2,Moderate Fraud Cost,Aggressive,299085


## 6. Final Decision Framework

The fraud model is designed as a risk-ranking and investigation-prioritization system rather than an automatic fraud-decision engine.

The operational workflow is:

1. Score each incoming application using the CatBoost fraud model.
2. Convert the raw model score into a calibrated fraud probability.
3. Assign an interpretable risk segment.
4. Rank applications by predicted fraud risk.
5. Select applications for manual review according to available investigation capacity.
6. Use SHAP explanations to support investigation and model transparency.
7. Periodically monitor calibration, fraud prevalence, review rate, and model performance for temporal drift.

For the portfolio demonstration, a 5% review-capacity policy is retained as the primary operating point because it captures approximately half of fraud cases while reviewing a small fraction of applications.

However, the cost-sensitive analysis shows that a more aggressive review strategy may be preferable when the cost of missed fraud is substantially higher than the cost of investigation.

## 7. Business Impact Summary

The final system demonstrates how an imbalanced fraud-classification model can be transformed into an operational decision-support framework.

Key results include:

- strong out-of-time ranking performance,
- calibrated probability estimates,
- meaningful Low / Medium / High / Critical risk segmentation,
- substantial fraud concentration within the highest-risk applications,
- approximately 50% fraud capture within a 5% review capacity,
- more than 10× lift over the underlying fraud prevalence at that operating point,
- interpretable SHAP-based explanations,
- capacity-dependent policy selection,
- and a cost-sensitive framework for future institution-specific optimization.

The system is intended to support investigators and risk teams, not to replace human judgment or automatically deny applications.